In [9]:
import logging
import os
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from IPython.display import clear_output, display
from moviepy import VideoFileClip

from openretina.data_io.hoefling_2024.constants import BADEN_TYPE_BOUNDARIES, RGC_GROUP_GROUP_ID_TO_CLASS_NAME
from openretina.data_io.hoefling_2024.stimuli import movies_from_pickle
from openretina.models.core_readout import load_core_readout_from_remote
from openretina.utils.file_utils import get_cache_directory, get_local_file_path, optionally_download_from_url
from openretina.utils.misc import CustomPrettyPrinter
from openretina.utils.plotting import (
    create_roi_animation,
    display_video,
    numpy_to_mp4_video,
    prepare_video_for_display,
    stitch_videos,
)
from openretina.models.core_readout import UnifiedCoreReadout


logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)  # to display logs in jupyter notebooks

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

pp = CustomPrettyPrinter(indent=4, max_lines=40)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
movie_stimulus_path = '/home/bethge/bkr618/openretina_cache/euler_lab/hoefling_2024/stimuli/rgc_natstim_72x64_joint_normalized_2024-10-11.pkl'

movie_stimuli = movies_from_pickle(movie_stimulus_path)

In [13]:
import os
import hydra
with hydra.initialize(config_path=os.path.join("..", "configs"), version_base="1.3"):
    cfg = hydra.compose(config_name="hoefling_2024_core_readout_low_res.yaml")

In [14]:
responses_path = "/home/bethge/bkr618/openretina_cache/data/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.h5"
from openretina.utils.h5_handling import load_h5_into_dict
from openretina.data_io.hoefling_2024.responses import filter_responses, make_final_responses

responses_dict = load_h5_into_dict(file_path=responses_path)

filtered_responses_dict = filter_responses(responses_dict, **cfg.quality_checks)

final_responses = make_final_responses(filtered_responses_dict, response_type="natural")

Loading HDF5 file contents:   0%|          | 0/2077 [00:00<?, ?item/s]

Original dataset contains 7863 neurons over 67 fields
 ------------------------------------ 
Dropped 0 fields that did not contain the target cell types (67 remaining)
Overall, dropped 3034 neurons of non-target cell types (-38.59%).
 ------------------------------------ 
Dropped 0 fields with quality indices below threshold (67 remaining)
Overall, dropped 980 neurons over quality checks (-20.29%).
 ------------------------------------ 
Dropped 0 fields with classifier confidences below 0.25
Overall, dropped 705 neurons with classifier confidences below 0.25 (-18.32%).
 ------------------------------------ 
 ------------------------------------ 
Final dataset contains 3144 neurons over 67 fields
Total number of cells dropped: 4719 (-60.02%)


Upsampling natural spikes traces to get final responses.:   0%|          | 0/67 [00:00<?, ?it/s]

In [15]:

from openretina.data_io.base import compute_data_info
data_info = compute_data_info(neuron_data_dictionary=final_responses, movies_dictionary=movie_stimuli)
n_neurons_dict = data_info["n_neurons_dict"]

In [17]:
file_path = "/home/bethge/bkr618/openretina_cache/model_checkpoints/20-11-2025_vivit_retina.ckpt"


model = UnifiedCoreReadout.load_from_checkpoint(file_path, map_location="cuda", n_neurons_dict = n_neurons_dict)

ConfigAttributeError: Key 'channels' is not in struct
    full_key: model.core.channels
    object_type=dict

In [4]:
# First, put the stimuli in a torch tensor, which is what the model expects.
stim = torch.Tensor(movie_stimuli.test_movie).to(model.device)

# Second, we need to select one of the many experimental sessions the model was trained on to visualize a response.
example_session = model.readout.sessions[0]
#"session_5_ventral2_20210929" # Can pick any number as long as it is in range

with torch.no_grad():
    predicted_response = model.forward(stim.unsqueeze(0), data_key=example_session)
predicted_response_numpy = predicted_response.squeeze().cpu().numpy()

In [ ]:
# Create a dropdown for neuron selection
neuron_selector = widgets.Dropdown(
    options=list(range(predicted_response_numpy.shape[1])),
    value=0,
    description="Neuron:",
)


# Define the plotting function
def plot_response(neuron_idx):
    plt.figure(figsize=(12, 6))
    plt.plot(predicted_response_numpy[:, neuron_idx])
    plt.xlabel("Time [frames]")
    plt.ylabel("Response [a.u.]")
    sns.despine()
    plt.show()


# Create an interactive widget
widgets.interactive(plot_response, neuron_idx=neuron_selector)

interactive(children=(Dropdown(description='Neuron:', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 1…

In [12]:
cnn_model = load_core_readout_from_remote(
    "hoefling_2024_base_high_res", device="cuda" if torch.cuda.is_available() else "cpu"
)

2025-11-20 09:54:28,162 - INFO - Fetching file list for open-retina/open-retina...
2025-11-20 09:54:28,432 - INFO - Found target file at /home/bethge/bkr618/openretina_cache/model_checkpoints/24-01-2025/hoefling_2024_base_high_res.ckpt
2025-11-20 09:57:53,416 - INFO - in_shape_readout=torch.Size([16, 120, 18, 16])


In [13]:
roi_mask = cnn_model.data_info["sessions_kwargs"][example_session]["roi_mask"]
roi_ids = cnn_model.data_info["sessions_kwargs"][example_session]["roi_ids"]
cell_types = cnn_model.data_info["sessions_kwargs"][example_session]["group_assignment"]


In [37]:
roi_animation = create_roi_animation(
    roi_mask=roi_mask, activity=predicted_response_numpy.T, roi_ids=roi_ids, max_activity=5, visualize_ids=True
)
numpy_to_mp4_video(roi_animation, fps=30)



Generating frames for ROI animation:   0%|          | 0/743 [00:00<?, ?it/s]